### ЗАДАЧА: Панель SLA-ребейтов по доставке по паттерну `MVC`

Команда logistics finance разбирает кейсы по SLA-ребейтам: если доставка опоздала,
клиенту или продавцу может быть положена компенсация, а к логистическому партнеру — применен штраф.
Нужно реализовать внутреннюю консольную панель по паттерну `MVC`.

Слои:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `shipment_id` — идентификатор отправления;
- `courier` — служба доставки;
- `promised_days` — обещанный срок доставки;
- `actual_days` — фактический срок доставки;
- `order_value` — стоимость заказа;
- `shipping_fee` — стоимость доставки;
- `penalty_rate` — ставка компенсации за каждый день просрочки;
- `delay_days` — число дней просрочки;
- `requested_rebate` — расчетная сумма ребейта;
- `approved_rebate` — согласованная сумма ребейта;
- `courier_penalty` — штраф, который будет предъявлен логистическому партнеру;
- `status` — статус кейса;
- `coordinator` — сотрудник, который ведет кейс;
- `customer_contacted` — связывались ли с клиентом;
- `decision` — финальное решение.

## Формулы

При создании кейса и после изменения одобренной суммы нужно считать:
- `delay_days = max(actual_days - promised_days, 0)`
- `requested_rebate = min(order_value * penalty_rate * delay_days, shipping_fee + order_value * 0.2)`
- `approved_rebate` при создании равно `0.0`
- `courier_penalty = approved_rebate * 0.7`
- все денежные значения округляются до 2 знаков.

## Статусы

- `new`
- `investigating`
- `customer_contacted`
- `ready_for_approval`
- `approved`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `coordinator` несуществующему кейсу;
- финальные кейсы (`approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `coordinator`;
- связаться с клиентом можно только из `investigating`;
- при контакте с клиентом поле `customer_contacted` должно стать `True`, а статус — `customer_contacted`;
- установить `approved_rebate` можно только из `investigating` или `customer_contacted`;
- `approved_rebate` не может быть меньше `0`;
- `approved_rebate` не может быть больше `requested_rebate`;
- после изменения `approved_rebate` нужно пересчитать `courier_penalty`;
- перевод в `ready_for_approval` возможен только из `investigating` или `customer_contacted`;
- перевод в `ready_for_approval` возможен только если `approved_rebate > 0`;
- завершить кейс как `approved` можно только из `ready_for_approval`;
- завершить кейс как `rejected` можно только из `ready_for_approval`, если `approved_rebate == 0`;
- `escalated` можно сделать только из `investigating`, `customer_contacted` или `ready_for_approval`;
- при финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать координатора;
- начинать расследование;
- отмечать контакт с клиентом;
- устанавливать `approved_rebate`;
- переводить кейс в `ready_for_approval`;
- завершать кейс как `approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | shipment_id | courier | promised_days | actual_days | delay_days | order_value | shipping_fee | requested_rebate | approved_rebate | courier_penalty | status | coordinator | customer_contacted | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_requested_rebate` — общая расчетная сумма ребейтов;
- `total_approved_rebate` — общая согласованная сумма;
- `total_courier_penalty` — общая сумма штрафов логистическому партнеру;
- `delayed_shipments` — количество кейсов, где `delay_days > 0`;
- `contacted_cases` — количество кейсов, где `customer_contacted == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести список кейсов и summary.

In [ ]:
initial_cases = [
    ("DR-100", "SHP-9901", "FastBox", 2, 5, 4200.0, 300.0, 0.03),
    ("DR-101", "SHP-9902", "QuickShip", 3, 3, 1800.0, 220.0, 0.02),
]

actions = [
    ("show",),
    ("investigate", "DR-100"),
    ("assign", "DR-100", "Olga"),
    ("investigate", "DR-100"),
    ("contact", "DR-100"),
    ("set_rebate", "DR-100", 180.0),
    ("ready", "DR-100"),
    ("approve", "DR-100", "rebate_sent_to_customer"),
    ("create", "DR-102", "SHP-9903", "CityRun", 1, 4, 2600.0, 180.0, 0.04),
    ("assign", "DR-102", "Max"),
    ("investigate", "DR-102"),
    ("set_rebate", "DR-102", 150.0),
    ("ready", "DR-102"),
    ("escalate", "DR-102", "courier_disputes_delay_window"),
    ("create", "DR-103", "SHP-9904", "ParcelWay", 2, 6, 5200.0, 450.0, 0.03),
    ("assign", "DR-103", "Ina"),
    ("investigate", "DR-103"),
    ("set_rebate", "DR-103", 0.0),
    ("ready", "DR-103"),
    ("reject", "DR-103", "no_customer_refund_required"),
    ("show",),
]